[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/spherical_synchronization/spherical_synchronization.ipynb)

# Spherical Synchronization

This example follows spherical synchronization, the Kuramoto model on a sphere: a population of directions, each turning at the same natural rate and each attracted towards the population's mean, gathers into one direction. However many directions there are, the whole population follows a single Möbius transformation of the sphere, as Lipton, Mirollo and Strogatz showed, so its state is one transformation with six degrees of freedom.

The algebra `x+y+z+t-` holds the sphere's conformal geometry. A point of the sphere is a null ray, an antivector, `direction + time`, and a circle is a cut, a vector, through which a point passes when `cut & point == 0`. A rotor moves points and cuts alike and keeps circles and their angles: it is the Möbius transformation, and one rotor carries the population and any grid of circles together. Its generator is the shared rotation less a boost towards the population's mean, `mean.commutator(time)`, the turn from time towards the mean, which on the sphere is a tangential attraction; the population feeds back into it as it moves.

In the vector notation of the Kuramoto model each direction moves as $\dot{\mathbf x}_i = \boldsymbol\omega \times \mathbf x_i + K\,(\bar{\mathbf x} - (\bar{\mathbf x}\cdot\mathbf x_i)\,\mathbf x_i)$, with $\bar{\mathbf x}$ the mean direction and $K$ the coupling.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, concatenate, stack
from examples.animation import save_figure, show_animation
from examples.math.spherical_synchronization import render

In [ ]:
# The algebra x+y+z+t- of the sphere's conformal geometry.
from examples.math.spherical_synchronization.core import ga

mv = NumpyContext(ga).multivector
# Circles are cuts of the sphere, vectors; points are null rays, antivectors.
Cut = ga.gatype.vector()
Point = ga.gatype.antivector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
# The rays' time direction, which pairs to one with `mv.t`; the dual carries the minus of t's square.
time = -mv.t.dual()                                                   # [] Point


def directions(points: Point) -> Point:
    """The unit directions on the sphere of null rays: each scaled to unit time, less its time."""
    return points / (mv.t & points) - time

## 1. A population on the sphere

The points sample a nearly uniform population with a small initial preference. The circles mark which directions travel together. Points are null rays, antivectors, and circles are cuts, vectors: a point lies on a circle when `cut & point == 0`.

In [ ]:
n_particles = 192
n_meridians = 4
n_parallels = 5
initial_bias = 0.24
golden_angle = np.pi * (3 - np.sqrt(5))
longitudes = golden_angle * np.arange(n_particles) % (2 * np.pi)
latitudes = np.arcsin(np.linspace(-1 + 1 / n_particles, 1 - 1 / n_particles, n_particles))
meridian_angles = np.linspace(0, np.pi, n_meridians, endpoint=False)
parallel_angles = np.linspace(-np.pi / 3, np.pi / 3, n_parallels)


In [ ]:
longitude = (-mv.xy * longitudes / 2).exp()                     # [particles] Rotor
latitude = (-mv.xz * latitudes / 2).exp()                       # [particles] Rotor
seeds = ((longitude * latitude) >> mv.x.dual()) + time          # [particles] Point

# Each cut intersects the unit sphere in a circle.
meridians = (-mv.xy * meridian_angles / 2).exp() >> mv.y         # [meridians] Cut
parallels = mv.z + mv.t * np.sin(parallel_angles)                # [parallels] Cut
grid = concatenate((meridians, parallels), axis=0)               # [circles] Cut
initial = (-mv.xt * initial_bias).exp()                         # Rotor

save_figure(render.draw(directions(initial >> seeds), initial >> grid), "spherical_synchronization");

## 2. Attraction through one transformation

Each direction turns towards the population's mean. On null rays this attraction is a boost, `mean.commutator(time)`, which combines with the shared rotation in one bivector. Recomputing the mean as the population moves supplies the feedback. The circles undergo exactly the same transformation as the points. This cell takes about ten seconds, mostly drawing the animation.

In [ ]:
coupling = 1.2
rotation_speed = 0.25
dt = 0.05
n_steps = 100
duration_ms = 60
rotation = -mv.xy * rotation_speed                              # Bivector


In [ ]:
def generator(rotor: Rotor) -> Bivector:
    """The shared rotation, less a boost towards the population's mean direction."""
    population = directions(rotor >> seeds)                     # [particles] Point
    mean = population.mean(axis=-1)                             # [] Point
    # Rescaling each null ray to the sphere removes the radial part of this attraction.
    return rotation - coupling * mean.commutator(time)          # [] Bivector


def motion(rotor: Rotor) -> Iterator[Rotor]:
    """The common transformation from the start and after each step, the generator taken halfway."""
    while True:
        yield rotor
        halfway = (generator(rotor) * (dt / 4)).exp() * rotor    # [] Rotor
        rotor = (generator(halfway) * (dt / 2)).exp() * rotor    # [] Rotor


rotors = stack(list(islice(motion(initial), n_steps + 1)))      # [steps + 1] Rotor
# One rotor carries the population and the grid of circles alike.
population = directions(rotors[:, None] >> seeds)               # [steps + 1, particles] Point
circles = rotors[:, None] >> grid                               # [steps + 1, circles] Cut
show_animation(render.frames(population, circles), "spherical_synchronization", duration_ms)

In [ ]:
# checks
# Every direction stays on the sphere, the circles keep passing through the same particles' images, and
# the population gathers.
np.testing.assert_allclose((population + time).squared().to_array(), 0, atol=1e-10)
spread = (population - population.mean(axis=-1)[:, None]).squared().mean(axis=-1).to_array()   # [steps + 1]
assert abs(spread[-1]) < 0.05 * abs(spread[0])

Particle colours stay with each particle; teal curves are the carried circles, with the far side faded. The population gathers while the grid keeps its circles and their intersection angles. Synchronization is approached by an invertible conformal deformation: distinct particles approach one another without merging at any finite time. The number of particles changes the mean calculation, but the evolving state remains one rotor. This reduction depends on the shared natural rotation.